# Web Scraping Workshop

## Proactieve Zorgbemiddeling: hoe lang wacht Nederland op zorg?

Sinds 2026 geldt **proactieve zorgbemiddeling**: je verzekeraar moet je *actief* helpen als je te lang wacht op zorg. Maar hoe erg zijn de wachttijden?

We scrapen wachttijden van **Zorgkaart Nederland**, vergelijken met de treeknormen, en bouwen een interactieve **Wachttijden Radar**.

### Wat gaan we doen?
1. Webpagina ophalen met `requests`
2. HTML parsen met `BeautifulSoup`
3. Wachttijden extraheren
4. Meerdere specialismen scrapen
5. Vergelijken met **treeknormen**
6. Interactieve **Wachttijden Radar** bouwen

In [ ]:
%pip install requests beautifulsoup4 lxml pandas plotly

## Achtergrond: treeknormen

Maximale wachttijden (de **treeknormen**):
- **Polikliniekbezoek**: max 4 weken (28 dagen)
- **Diagnostisch onderzoek**: max 4 weken (28 dagen)
- **Behandeling**: max 7 weken (49 dagen)

Overschrijding? Dan moet je verzekeraar je *proactief* elders plaatsen.

## Stap 1: Een wachttijdenpagina ophalen

We halen een pagina op van **Zorgkaart Nederland** met wachttijden voor cardiologie.

In [ ]:
import requests

url = "https://www.zorgkaartnederland.nl/wachttijden/cardiologie-2"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)

print(f"Status code: {response.status_code}")  # 200 = succes!
print(f"Grootte: {len(response.text):,} tekens")
print(f"\nEerste 500 tekens:\n{response.text[:500]}")

## Stap 2: HTML parsen met BeautifulSoup

BeautifulSoup parsed HTML en laat ons zoeken met CSS selectors.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "lxml")

table = soup.select_one("table.table-certificates")
print(f"Tabel gevonden: {table is not None}")

rows = table.select("tr")
print(f"Aantal rijen: {len(rows)} (1 header + {len(rows)-1} ziekenhuizen)")

header = [th.get_text(strip=True) for th in rows[0].select("th")]
print(f"Kolommen: {header}")

print("\n--- Eerste 3 rijen ---")
for row in rows[1:4]:
    cells = [td.get_text(strip=True) for td in row.select("td")]
    print(cells)

## Stap 3: Data uit de tabel halen

Elke rij heeft 3 kolommen: zorgaanbieder, locatie, wachttijd (dagen).

In [ ]:
row = rows[1]  # eerste data-rij
cells = row.select("td")

naam = cells[0].get_text(strip=True)
print(f"Naam: {naam}")

link_el = cells[0].select_one("a")
profiel_url = link_el["href"] if link_el else ""
print(f"Profiel: https://www.zorgkaartnederland.nl{profiel_url}")

locatie = cells[1].get_text(strip=True)
print(f"Locatie: {locatie}")

wachttijd_tekst = cells[2].get_text(strip=True)
print(f"Wachttijd: {wachttijd_tekst} dagen")

## Stap 4: Alle ziekenhuizen van één specialisme

We stoppen de logica in een functie.

In [ ]:
def scrape_wachttijden(url):
    """Scrape alle wachttijden van één Zorgkaart Nederland-pagina."""
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "lxml")
    
    table = soup.select_one("table.table-certificates")
    if not table:
        return []
    
    resultaten = []
    for row in table.select("tr")[1:]:  # skip header
        cells = row.select("td")
        if len(cells) < 3:
            continue
        
        link_el = cells[0].select_one("a")
        wachttijd_tekst = cells[2].get_text(strip=True)
        
        resultaten.append({
            "zorgaanbieder": cells[0].get_text(strip=True),
            "locatie": cells[1].get_text(strip=True),
            "wachttijd_dagen": int(wachttijd_tekst) if wachttijd_tekst.isdigit() else None,
            "profiel_url": "https://www.zorgkaartnederland.nl" + link_el["href"] if link_el else "",
        })
    
    return resultaten

resultaten = scrape_wachttijden("https://www.zorgkaartnederland.nl/wachttijden/cardiologie-2")
print(f"Gevonden: {len(resultaten)} ziekenhuizen")
resultaten[:3]

## Stap 5: Meerdere specialismen scrapen

We scrapen 12 specialismen (poliklinieken, diagnostiek, behandelingen).

In [ ]:
import time
import pandas as pd

specialismen = {
    "cardiologie-2": ("Cardiologie", "polikliniek"),
    "chirurgie-heelkunde": ("Chirurgie", "polikliniek"),
    "dermatologie": ("Dermatologie", "polikliniek"),
    "neurologie": ("Neurologie", "polikliniek"),
    "oogheelkunde": ("Oogheelkunde", "polikliniek"),
    "orthopedie-2": ("Orthopedie", "polikliniek"),
    "ct": ("CT-scan", "diagnostiek"),
    "echografie-radiologie": ("Echografie", "diagnostiek"),
    "mri-radiologie": ("MRI", "diagnostiek"),
    "staaroperatie-oogheelkunde": ("Staaroperatie", "behandeling"),
    "totale-knievervanging-knieprothese-orthopedie": ("Knievervanging", "behandeling"),
    "totale-heupvervanging-heupprothese-orthopedie": ("Heupvervanging", "behandeling"),
}

alle_data = []
for slug, (naam, wtype) in specialismen.items():
    url = f"https://www.zorgkaartnederland.nl/wachttijden/{slug}"
    resultaten = scrape_wachttijden(url)
    for r in resultaten:
        r["specialisme"] = naam
        r["type"] = wtype
    alle_data.extend(resultaten)
    print(f"  {naam}: {len(resultaten)} ziekenhuizen")
    time.sleep(0.5)  # wees netjes

df = pd.DataFrame(alle_data)
print(f"\nTotaal: {len(df)} rijen, {df['specialisme'].nunique()} specialismen")
df.head(10)

## Stap 6: Treeknorm-analyse

We vergelijken de werkelijke wachttijden met de treeknormen.

In [ ]:
treeknormen = {
    "polikliniek": 28,   # 4 weken
    "diagnostiek": 28,   # 4 weken
    "behandeling": 49,   # 7 weken
}

df["treeknorm_dagen"] = df["type"].map(treeknormen)

df["overschrijding"] = df["wachttijd_dagen"] > df["treeknorm_dagen"]

n_totaal = len(df.dropna(subset=["wachttijd_dagen"]))
n_overschrijding = df["overschrijding"].sum()
pct = n_overschrijding / n_totaal * 100

print(f"{n_overschrijding} van {n_totaal} ziekenhuizen ({pct:.0f}%) overschrijdt de treeknorm!")
print(f"\nPer specialisme:")
print(df.groupby("specialisme")["overschrijding"].mean().mul(100).round(0).sort_values(ascending=False).to_string())

In [ ]:
print("De 15 langste wachttijden in onze dataset:\n")
top15 = df.nlargest(15, "wachttijd_dagen")[["specialisme", "zorgaanbieder", "locatie", "wachttijd_dagen", "treeknorm_dagen"]].copy()
top15["overschrijding_dagen"] = top15["wachttijd_dagen"] - top15["treeknorm_dagen"]
print(top15.to_string(index=False))

## Stap 7: Visualiseren

In [ ]:
import plotly.express as px

pct_per_spec = (
    df.groupby(["specialisme", "type"])["overschrijding"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .rename(columns={"overschrijding": "pct_overschrijding"})
    .sort_values("pct_overschrijding", ascending=True)
)

fig = px.bar(
    pct_per_spec,
    x="pct_overschrijding",
    y="specialisme",
    color="type",
    orientation="h",
    title="Percentage ziekenhuizen dat de treeknorm overschrijdt",
    labels={"pct_overschrijding": "% overschrijding", "specialisme": "", "type": "Type"},
    color_discrete_map={"polikliniek": "#2196F3", "diagnostiek": "#FF9800", "behandeling": "#9C27B0"},
)
fig.add_vline(x=50, line_dash="dash", line_color="red", annotation_text="50%")
fig.update_layout(height=500)
fig.show()

### Gemiddelde wachttijd

In [ ]:
gem = (
    df.groupby(["specialisme", "type"])
    .agg(gem_wachttijd=("wachttijd_dagen", "mean"), treeknorm=("treeknorm_dagen", "first"))
    .reset_index()
    .sort_values("gem_wachttijd", ascending=True)
)

fig = px.bar(
    gem,
    x="gem_wachttijd",
    y="specialisme",
    orientation="h",
    title="Gemiddelde wachttijd per specialisme (in dagen)",
    labels={"gem_wachttijd": "Gemiddelde wachttijd (dagen)", "specialisme": ""},
    color="type",
    color_discrete_map={"polikliniek": "#2196F3", "diagnostiek": "#FF9800", "behandeling": "#9C27B0"},
)

fig.add_vline(x=28, line_dash="dash", line_color="#F44336", annotation_text="Treeknorm poli/diag (28d)")
fig.add_vline(x=49, line_dash="dash", line_color="#F44336", annotation_text="Treeknorm behandeling (49d)")
fig.update_layout(height=500)
fig.show()

### Spreiding

In [ ]:
fig = px.box(
    df,
    x="wachttijd_dagen",
    y="specialisme",
    color="type",
    orientation="h",
    title="Spreiding wachttijden per specialisme",
    labels={"wachttijd_dagen": "Wachttijd (dagen)", "specialisme": ""},
    color_discrete_map={"polikliniek": "#2196F3", "diagnostiek": "#FF9800", "behandeling": "#9C27B0"},
)
fig.add_vline(x=28, line_dash="dash", line_color="#F44336", opacity=0.5)
fig.add_vline(x=49, line_dash="dash", line_color="#F44336", opacity=0.5)
fig.update_layout(height=600)
fig.show()

## Stap 8: Interactieve Wachttijden Radar

Het eindproduct: kies een specialisme, zie welke ziekenhuizen de treeknorm overschrijden.

In [ ]:
import plotly.graph_objects as go

specialisme_namen = sorted(df["specialisme"].unique())
fig = go.Figure()

for i, spec in enumerate(specialisme_namen):
    subset = df[df["specialisme"] == spec].dropna(subset=["wachttijd_dagen"]).sort_values("wachttijd_dagen")
    tk = subset["treeknorm_dagen"].iloc[0]
    kleuren = ["#4CAF50" if w <= tk else "#F44336" for w in subset["wachttijd_dagen"]]
    labels = [f"{r.zorgaanbieder} ({r.locatie})" for _, r in subset.iterrows()]
    fig.add_trace(go.Bar(x=subset["wachttijd_dagen"].values, y=labels, orientation="h",
                         marker_color=kleuren, visible=(i == 0), name=spec))

buttons = []
for i, spec in enumerate(specialisme_namen):
    sub = df[df["specialisme"] == spec]
    tk = sub["treeknorm_dagen"].iloc[0]
    n_over = (sub["wachttijd_dagen"] > tk).sum()
    vis = [False] * len(specialisme_namen)
    vis[i] = True
    buttons.append(dict(label=spec, method="update", args=[
        {"visible": vis},
        {"title": f"Wachttijden Radar: {spec}<br><sub>{n_over}/{len(sub)} overschrijdt treeknorm ({tk}d)</sub>",
         "shapes": [dict(type="line", x0=tk, x1=tk, y0=-0.5, y1=len(sub)-0.5,
                         line=dict(color="#FF9800", width=2, dash="dash"))]}]))

first = df[df["specialisme"] == specialisme_namen[0]]
ftk = first["treeknorm_dagen"].iloc[0]
fig.update_layout(
    title=f"Wachttijden Radar: {specialisme_namen[0]}<br><sub>{(first['wachttijd_dagen']>ftk).sum()}/{len(first)} overschrijdt treeknorm ({ftk}d)</sub>",
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True, x=0, xanchor="left", y=1.15, yanchor="top")],
    xaxis_title="Wachttijd (dagen)", height=max(600, len(first)*18), margin=dict(l=300),
    shapes=[dict(type="line", x0=ftk, x1=ftk, y0=-0.5, y1=len(first)-0.5, line=dict(color="#FF9800", width=2, dash="dash"))])
fig.show()

In [ ]:
fig.write_html("wachttijden_radar.html", include_plotlyjs=True)
print("Opgeslagen: wachttijden_radar.html")
print("Open dit bestand in je browser voor de interactieve Wachttijden Radar!")

df.to_csv("wachttijden_zorg.csv", index=False)
print(f"\nData opgeslagen: wachttijden_zorg.csv ({len(df)} rijen)")

## Recap

- **Webpagina ophalen**: `requests.get(url)`
- **HTML parsen**: `BeautifulSoup(html, "lxml")`
- **Tabel zoeken**: `soup.select_one("table.table-certificates")`
- **Rijen doorlopen**: `table.select("tr")[1:]`
- **Meerdere pagina's**: Loop + `time.sleep()`
- **Data structureren**: `pd.DataFrame(lijst_van_dicts)`
- **Interactieve viz**: `plotly` + `fig.write_html()`

### Zelf verder?
- Meer specialismen: [/wachttijden/poliklinieken](https://www.zorgkaartnederland.nl/wachttijden/poliklinieken)
- De overzichtspagina scrapen om alle links automatisch te vinden
- Per ziekenhuis de detailpagina scrapen